# 从零训练迷你英中翻译模型 (Seq2Seq - 长句/双子句瓶颈对比实验)

完全复现经典实验：用连接词 and/和 将短句拼接为长句/双子句（最长 32 Token），在 7,440 组长句数据集上对比【无 Attention 模型】与【带 Attention 模型】。


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
import itertools
import torch.nn.functional as F
import sys

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


## 1. 动态生成 7440 组包含单子句与双子句的数据集

In [2]:
def build_dataset():
    animals = {"duck": "鸭子", "cat": "猫", "dog": "狗", "cow": "牛", "bird": "鸟", "horse": "马", "bear": "熊", "lion": "狮子"}
    colors = {"quiet": "安静", "red": "红色", "black": "黑色", "white": "白色", "blue": "蓝色", "green": "绿色", "yellow": "黄色", "brown": "棕色"}
    locations = {"next to": "旁边", "behind": "后面", "in front of": "前面", "on": "上面", "under": "下面", "near": "附近"}
    objects = {"door": "门", "tree": "树", "table": "桌子", "chair": "椅子", "box": "盒子", "window": "窗户", "car": "汽车", "apple": "苹果"}
    names = {"alice": "爱丽丝", "bob": "鲍勃", "charlie": "查理", "david": "大卫", "emma": "艾玛", "fiona": "菲奥娜", "george": "乔治", "henry": "亨利"}
    numbers = {1: ("a", "one", "一"), 2: ("two", "two", "两"), 3: ("three", "three", "三"), 4: ("four", "four", "四"), 5: ("five", "five", "五"), 6: ("six", "six", "六")}

    pattern1_all = []
    for num, color, animal, loc, obj in itertools.product(numbers.keys(), colors.keys(), animals.keys(), locations.keys(), objects.keys()):
        en_num = numbers[num][0]
        zh_num = numbers[num][2]
        if num == 1:
            en = f"there is {en_num} {color} {animal} {loc} the {obj}"
        else:
            en = f"there are {en_num} {color} {animal}s {loc} the {obj}"
        zh = f"{objects[obj]} {locations[loc]} 有 {zh_num} 只 {colors[color]} {animals[animal]}"
        pattern1_all.append((en, zh))

    pattern2_all = []
    for name, num, color, obj in itertools.product(names.keys(), numbers.keys(), colors.keys(), objects.keys()):
        en_num = numbers[num][1]
        zh_num = numbers[num][2]
        if num == 1:
            en = f"{name} has {en_num} {color} {obj}"
        else:
            en = f"{name} has {en_num} {color} {obj}s"
        zh = f"{names[name]} 有 {zh_num} 个 {colors[color]} {objects[obj]}"
        pattern2_all.append((en, zh))

    random.seed(42)
    random.shuffle(pattern1_all)
    random.shuffle(pattern2_all)
    
    # 3720 单子句 + 3720 用 and/和 拼接的双子句 = 7440 组长句数据
    singles = pattern1_all[:1860] + pattern2_all[:1860]
    random.shuffle(singles)
    
    doubles = []
    for i in range(3720):
        s1 = singles[i]
        s2 = singles[(i + 1860) % 3720]
        en_d = f"{s1[0]} and {s2[0]}"
        zh_d = f"{s1[1]} 和 {s2[1]}"
        doubles.append((en_d, zh_d))
        
    all_pairs = singles + doubles
    random.shuffle(all_pairs)
    return all_pairs

data_pairs = build_dataset()

# 8:1:1 划分
train_size = int(len(data_pairs) * 0.8)
val_size = int(len(data_pairs) * 0.1)
train_data = data_pairs[:train_size]
val_data = data_pairs[train_size:train_size+val_size]
test_data = data_pairs[train_size+val_size:]

print(f"数据总量: {len(data_pairs)} (Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)})")

# 构建词表
en_vocab = {"<PAD>":0, "<BOS>":1, "<EOS>":2, "<UNK>":3}
zh_vocab = {"<PAD>":0, "<BOS>":1, "<EOS>":2, "<UNK>":3}
for en, zh in data_pairs:
    for word in en.split():
        if word not in en_vocab: en_vocab[word] = len(en_vocab)
    for word in zh.split():
        if word not in zh_vocab: zh_vocab[word] = len(zh_vocab)

en_idx2word = {v: k for k, v in en_vocab.items()}
zh_idx2word = {v: k for k, v in zh_vocab.items()}
print(f"EN Vocab Size: {len(en_vocab)}, ZH Vocab Size: {len(zh_vocab)}")


数据总量: 7440 (Train: 5952, Val: 744, Test: 744)
EN Vocab Size: 74, ZH Vocab Size: 52


## 2. 定义 Dataset 与 DataLoader (统一 Padding 长度为 32)

In [3]:
PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3
MAX_PAD_LEN = 32  # 适应最长 32 Token 的长句/双子句

class TranslationDataset(Dataset):
    def __init__(self, data_pairs, en_vocab, zh_vocab):
        self.data_pairs = data_pairs
        self.en_vocab = en_vocab
        self.zh_vocab = zh_vocab
        
    def __len__(self):
        return len(self.data_pairs)
    
    def __getitem__(self, idx):
        en, zh = self.data_pairs[idx]
        en_indices = [self.en_vocab.get(w, UNK_IDX) for w in en.split()]
        zh_indices = [self.zh_vocab.get(w, UNK_IDX) for w in zh.split()]
        
        pad_len = max(0, MAX_PAD_LEN - len(en_indices))
        eng_padded = [PAD_IDX] * pad_len + en_indices
        chn_target = [BOS_IDX] + zh_indices + [EOS_IDX]
        
        return torch.tensor(eng_padded, dtype=torch.long), torch.tensor(chn_target, dtype=torch.long)

def collate_fn(batch):
    eng_batch, chn_batch = zip(*batch)
    eng_padded = torch.nn.utils.rnn.pad_sequence(eng_batch, batch_first=True, padding_value=PAD_IDX)
    chn_padded = torch.nn.utils.rnn.pad_sequence(chn_batch, batch_first=True, padding_value=PAD_IDX)
    return eng_padded, chn_padded

train_dataset = TranslationDataset(train_data, en_vocab, zh_vocab)
val_dataset = TranslationDataset(val_data, en_vocab, zh_vocab)
test_dataset = TranslationDataset(test_data, en_vocab, zh_vocab)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, collate_fn=collate_fn)


## 3. 模型架构定义 (无 Attention 模型 vs 带矩阵 Attention 模型)

In [4]:
# --------------- 1. 无 Attention 架构 --------------- 
class EncoderNoAttn(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super(EncoderNoAttn, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded)
        return output, hidden

class DecoderNoAttn(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super(DecoderNoAttn, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded, hidden)
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden

class Seq2SeqNoAttn(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2SeqNoAttn, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, target, teacher_forcing_ratio=0.5):
        batch_size = source.shape[0]
        target_len = target.shape[1]
        target_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, target_len, target_vocab_size).to(self.device)
        _, hidden = self.encoder(source)
        x = target[:, 0].unsqueeze(1)
        for t in range(1, target_len):
            prediction, hidden = self.decoder(x, hidden)
            outputs[:, t, :] = prediction
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)
            x = target[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1)
        return outputs

# --------------- 2. 带 Attention (Matrix) 架构 --------------- 
class EncoderAttn(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super(EncoderAttn, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded)
        return output, hidden

class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.linear_in = nn.Linear(hidden_size, hidden_size)

    def forward(self, hidden, encoder_outputs):
        if hidden.dim() == 2: hidden_q = hidden.unsqueeze(1)
        elif hidden.shape[0] == 1 and hidden.dim() == 3: hidden_q = hidden.transpose(0, 1)
        else: hidden_q = hidden
        q_proj = self.linear_in(hidden_q)
        scores = torch.bmm(q_proj, encoder_outputs.transpose(1, 2))
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.bmm(attn_weights, encoder_outputs)
        return context, attn_weights

class DecoderAttn(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super(DecoderAttn, self).__init__()
        self.attention = Attention(hidden_size)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size + hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size + hidden_size, vocab_size)

    def forward(self, x, hidden, encoder_outputs):
        embedded = self.embedding(x)
        context, attn_weights = self.attention(hidden, encoder_outputs)
        gru_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.gru(gru_input, hidden)
        output_combined = torch.cat((output, context), dim=2)
        prediction = self.fc(output_combined.squeeze(1))
        return prediction, hidden, attn_weights

class Seq2SeqAttn(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2SeqAttn, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, target, teacher_forcing_ratio=0.5):
        batch_size = source.shape[0]
        target_len = target.shape[1]
        target_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, target_len, target_vocab_size).to(self.device)
        encoder_outputs, hidden = self.encoder(source)
        x = target[:, 0].unsqueeze(1)
        for t in range(1, target_len):
            prediction, hidden, _ = self.decoder(x, hidden, encoder_outputs)
            outputs[:, t, :] = prediction
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)
            x = target[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1)
        return outputs


## 4. 训练与测试函数

In [5]:
ENG_VOCAB_SIZE = len(en_vocab)
CHN_VOCAB_SIZE = len(zh_vocab)
EMBED_SIZE = 64
HIDDEN_SIZE = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def train_epoch(model, dataloader, optimizer, criterion, device, clip=1.0):
    model.train()
    epoch_loss = 0
    for source, target in dataloader:
        source = source.to(device)
        target = target.to(device)
        optimizer.zero_grad()
        outputs = model(source, target, teacher_forcing_ratio=0.5)
        outputs = outputs[:, 1:].contiguous().view(-1, outputs.shape[-1])
        target = target[:, 1:].contiguous().view(-1)
        loss = criterion(outputs, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for source, target in dataloader:
            source = source.to(device)
            target = target.to(device)
            outputs = model(source, target, teacher_forcing_ratio=0.0)
            outputs = outputs[:, 1:].contiguous().view(-1, outputs.shape[-1])
            target = target[:, 1:].contiguous().view(-1)
            loss = criterion(outputs, target)
            epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

def translate_no_attn(model, english_text, device, max_len=40):
    model.eval()
    with torch.no_grad():
        en_words = english_text.split()
        english_indices = [en_vocab.get(w, UNK_IDX) for w in en_words]
        pad_len = max(0, MAX_PAD_LEN - len(english_indices))
        eng_padded = [PAD_IDX] * pad_len + english_indices
        english_tensor = torch.tensor(eng_padded, dtype=torch.long).unsqueeze(0).to(device)
        _, hidden = model.encoder(english_tensor)
        x = torch.tensor([[BOS_IDX]], dtype=torch.long).to(device)
        predicted_words = []
        for _ in range(max_len):
            prediction, hidden = model.decoder(x, hidden)
            top1 = prediction.argmax(1).item()
            if top1 == EOS_IDX: break
            predicted_words.append(top1)
            x = torch.tensor([[top1]], dtype=torch.long).to(device)
    return predicted_words

def translate_attn(model, english_text, device, max_len=40):
    model.eval()
    with torch.no_grad():
        en_words = english_text.split()
        english_indices = [en_vocab.get(w, UNK_IDX) for w in en_words]
        pad_len = max(0, MAX_PAD_LEN - len(english_indices))
        eng_padded = [PAD_IDX] * pad_len + english_indices
        english_tensor = torch.tensor(eng_padded, dtype=torch.long).unsqueeze(0).to(device)
        encoder_outputs, hidden = model.encoder(english_tensor)
        x = torch.tensor([[BOS_IDX]], dtype=torch.long).to(device)
        predicted_words = []
        for _ in range(max_len):
            prediction, hidden, _ = model.decoder(x, hidden, encoder_outputs)
            top1 = prediction.argmax(1).item()
            if top1 == EOS_IDX: break
            predicted_words.append(top1)
            x = torch.tensor([[top1]], dtype=torch.long).to(device)
    return predicted_words


## 5. 执行对比训练与 744 条长句测试集评估

In [6]:
# ==================== 1. 训练 模型 A（无 Attention） ====================
print("================ 开始训练 模型 A（无 Attention） ================")
enc_no_attn = EncoderNoAttn(ENG_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)
dec_no_attn = DecoderNoAttn(CHN_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)
model_no_attn = Seq2SeqNoAttn(enc_no_attn, dec_no_attn, DEVICE).to(DEVICE)
opt_no_attn = optim.Adam(model_no_attn.parameters(), lr=0.002)

best_val_loss = float("inf")
early_stop = 0
for epoch in range(50):
    tr_l = train_epoch(model_no_attn, train_loader, opt_no_attn, criterion, DEVICE)
    val_l = evaluate(model_no_attn, val_loader, criterion, DEVICE)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"模型 A Epoch [{epoch+1}/50], Train Loss: {tr_l:.4f}, Val Loss: {val_l:.4f}")
    if val_l < best_val_loss:
        best_val_loss = val_l
        early_stop = 0
        torch.save(model_no_attn.state_dict(), "model_no_attn_long.pt")
    else:
        early_stop += 1
        if early_stop >= 5:
            print(f"模型 A 在 Epoch {epoch+1} 触发 Early Stopping")
            break

model_no_attn.load_state_dict(torch.load("model_no_attn_long.pt"))

# ==================== 2. 训练 模型 B（带 Attention） ====================
print("\n================ 开始训练 模型 B（带 Attention） ================")
enc_attn = EncoderAttn(ENG_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)
dec_attn = DecoderAttn(CHN_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)
model_attn = Seq2SeqAttn(enc_attn, dec_attn, DEVICE).to(DEVICE)
opt_attn = optim.Adam(model_attn.parameters(), lr=0.002)

best_val_loss = float("inf")
early_stop = 0
for epoch in range(50):
    tr_l = train_epoch(model_attn, train_loader, opt_attn, criterion, DEVICE)
    val_l = evaluate(model_attn, val_loader, criterion, DEVICE)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"模型 B Epoch [{epoch+1}/50], Train Loss: {tr_l:.4f}, Val Loss: {val_l:.4f}")
    if val_l < best_val_loss:
        best_val_loss = val_l
        early_stop = 0
        torch.save(model_attn.state_dict(), "model_attn_long.pt")
    else:
        early_stop += 1
        if early_stop >= 5:
            print(f"模型 B 在 Epoch {epoch+1} 触发 Early Stopping")
            break

model_attn.load_state_dict(torch.load("model_attn_long.pt"))

# ==================== 3. 测试集评测与对比 ====================
print("\n================ 744条长句/双子句测试集评测结果 ================")
corr_no_attn = 0
corr_attn = 0
total_test = len(test_data)

for k in range(total_test):
    sample_en, sample_zh = test_data[k]
    pred_no_attn_idx = translate_no_attn(model_no_attn, sample_en, DEVICE)
    pred_no_attn_zh = " ".join([zh_idx2word.get(i, "<UNK>") for i in pred_no_attn_idx])
    if pred_no_attn_zh == sample_zh:
        corr_no_attn += 1
        
    pred_attn_idx = translate_attn(model_attn, sample_en, DEVICE)
    pred_attn_zh = " ".join([zh_idx2word.get(i, "<UNK>") for i in pred_attn_idx])
    if pred_attn_zh == sample_zh:
        corr_attn += 1

acc_no_attn = (corr_no_attn / total_test) * 100
acc_attn = (corr_attn / total_test) * 100

print(f"测试集样本总数: {total_test}")
print(f"【无 Attention 模型 A】 正确数: {corr_no_attn} / {total_test}, 句级准确率: {acc_no_attn:.2f}%")
print(f"【带 Attention 模型 B】 正确数: {corr_attn} / {total_test}, 句级准确率: {acc_attn:.2f}%")

print("\n---------------- 展示双子句测试样例对比 ----------------")
double_samples_shown = 0
for k in range(total_test):
    sample_en, sample_zh = test_data[k]
    if "and" in sample_en and double_samples_shown < 5:
        double_samples_shown += 1
        p_no = " ".join([zh_idx2word.get(i, "<UNK>") for i in translate_no_attn(model_no_attn, sample_en, DEVICE)])
        p_att = " ".join([zh_idx2word.get(i, "<UNK>") for i in translate_attn(model_attn, sample_en, DEVICE)])
        print(f"双子句样例 [{double_samples_shown}]")
        print("英文输入:", sample_en)
        print("目标中文:", sample_zh)
        print("无 Attention 预测:", p_no, " (正确)" if p_no == sample_zh else " (错误/瓶颈丢失)")
        print("带 Attention 预测:", p_att, " (正确)" if p_att == sample_zh else " (错误)")
        print("-" * 50)


================ 开始训练 模型 A（无 Attention） ================
模型 A Epoch [1/50], Train Loss: 3.1122, Val Loss: 2.5924
模型 A Epoch [10/50], Train Loss: 0.9460, Val Loss: 0.9278
模型 A Epoch [20/50], Train Loss: 0.7255, Val Loss: 0.7573
模型 A Epoch [30/50], Train Loss: 0.5310, Val Loss: 0.5276
模型 A Epoch [40/50], Train Loss: 0.3227, Val Loss: 0.3391
模型 A Epoch [50/50], Train Loss: 0.2128, Val Loss: 0.2687

================ 开始训练 模型 B（带 Attention） ================
模型 B Epoch [1/50], Train Loss: 3.1380, Val Loss: 2.5372
模型 B Epoch [10/50], Train Loss: 0.1346, Val Loss: 0.1136
模型 B Epoch [20/50], Train Loss: 0.0783, Val Loss: 0.1013
模型 B Epoch [30/50], Train Loss: 0.0342, Val Loss: 0.0415
模型 B Epoch [40/50], Train Loss: 0.0166, Val Loss: 0.0337
模型 B Epoch [50/50], Train Loss: 0.0035, Val Loss: 0.0030

================ 744条长句/双子句测试集评测结果 ================
测试集样本总数: 744
【无 Attention 模型 A】 正确数: 355 / 744, 句级准确率: 47.72%
【带 Attention 模型 B】 正确数: 723 / 744, 句级准确率: 97.18%

---------------- 展示双子句测试样例对比 ---------